In [ ]:
import os
import csv
import cv2
import imageio
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, Image as IPImage
from tqdm.auto import tqdm


In [ ]:
# Input frames.
# frame_dir = "/Volumes/PortableSSD/tutrtletest/turtle_test_2_short/frames"
# frame_dir = "/Volumes/PortableSSD/tutrtletest/turtlepond_2026-06-29_111037/frames"
frame_dir = "/Volumes/PortableSSD/tutrtletest/turtlepond_2026-06-29_112336/frames"

num_frames = 40
first_frame_number = 1000
last_frame_number = 2000
# last_frame_number = -1000
frame_selection_mode = "evenly_sampled"  # "first" or "evenly_sampled"
background_sample_count = 1000
frames_to_plot = 6

# Output folders and filenames.
output_dir = "outputs"
diagnostics_dir = os.path.join(output_dir, "diagnostics")
frame_source_name = os.path.basename(os.path.normpath(frame_dir))
if frame_source_name.lower() == "frames":
    frame_source_name = os.path.basename(os.path.dirname(os.path.normpath(frame_dir)))
frame_source_tag = "".join(
    char if char.isalnum() or char in "-_" else "_"
    for char in frame_source_name
).strip("_") or "frames"
frame_selection_tag = f"{first_frame_number}_to_{last_frame_number}_{frame_selection_mode}"
output_tag = f"{frame_source_tag}_warped_sort_{frame_selection_tag}_{num_frames}_frames"
tracked_gif_name = f"tracked_blobs_{output_tag}.gif"
tracked_raw_gif_name = f"tracked_blobs_raw_{output_tag}.gif"
tracked_echogram_name = f"tracked_echogram_tol_{output_tag}.png"
tracked_echogram_objects_name = f"tracked_echogram_objects_{output_tag}.png"
tracked_raw_echogram_name = f"tracked_raw_echogram_{output_tag}.png"
tracked_raw_echogram_objects_name = f"tracked_raw_echogram_objects_{output_tag}.png"
tracking_csv_name = f"tracking_data_{output_tag}.csv"
track_length_bar_name = f"track_length_by_id_{output_tag}.png"
track_area_bar_name = f"track_area_by_id_{output_tag}.png"
track_speed_bar_name = f"track_speed_by_id_{output_tag}.png"
track_length_area_scatter_name = f"track_length_vs_area_{output_tag}.png"
tracked_echogram_brightness_name = f"tracked_echogram_tol_brightness_{output_tag}.png"
tracked_raw_echogram_brightness_name = f"tracked_raw_echogram_brightness_{output_tag}.png"
tracked_echogram_masks_name = f"tracked_echogram_masks_{output_tag}.gif"
preview_frames_gif_name = f"frames_{output_tag}.gif"
preview_bgs_gif_name = f"positive_bgs_{output_tag}.gif"
preview_bgs_tolled_gif_name = f"positive_bgs_tolled_{output_tag}.gif"
preview_tinted_gif_name = f"frames_with_tolled_bgs_tint_{output_tag}.gif"

# Polar-to-cartesian warp.
center_column_signal_threshold = 8
warp_interpolation_mode = "nearest"
warp_output_scale = 5

# Background subtraction.
threshold_mode = "percentile"  # "fixed", "percentile", or "adaptive"
bgs_tol = 0.15
bgs_percentile = 95
adaptive_threshold_block_size = 51
adaptive_threshold_c = -10
adaptive_min_bgs_tol = 0.07
processed_frame_blur_kernel_size = 15

# GIF rendering.
gif_duration = 0.2
motion_tint_color = np.array([255, 80, 20], dtype=np.float32)
motion_tint_alpha = 0.45

# Detection and SORT tracker configuration.
min_blob_area = 30
cluster_distance = 30
morph_kernel_size = 5

sort_max_age = 20
sort_min_hits = 3
sort_iou_threshold = 0.01
sort_exact_assignment_limit = 18
sort_collision_tiebreaker_method = "iou"  # "iou", "confidence", "area", or "trajectory"
sort_collision_iou_threshold = 0.01
sort_collision_iou_tiebreaker = 0.01
sort_trajectory_max_distance = 120
sort_trajectory_min_score = 0.15
sort_trajectory_iou_tiebreaker = 0.03
sort_keep_collision_tracks_alive = True

min_track_frames = 6  # Reject short non-large tracks.
reject_tiny_flicker_tracks = True
min_track_median_area = 90
min_large_detection_fraction = 0.35
large_detection_area = 120
reject_short_flash_tracks = True
max_short_flash_frames = 15
max_short_flash_median_area = 3000
max_short_flash_area = 10000

# Stable ID colors used by GIFs and echograms.
track_color_palette = [
    (213, 94, 0),
    (0, 158, 115),
    (240, 228, 66),
    (0, 114, 178),
    (230, 159, 0),
    (86, 180, 233),
    (204, 121, 0),
    (0, 150, 136),
    (178, 34, 34),
    (46, 139, 87),
    (65, 105, 225),
    (218, 165, 32),
    (112, 128, 144),
    (128, 0, 0),
    (60, 179, 113),
    (0, 0, 128),
    (255, 140, 0),
    (47, 79, 79),
    (154, 205, 50),
    (70, 130, 180),
]
unassigned_segmentation_color = (255, 0, 255)
segmentation_overlay_alpha = 0.65


# Echogram rendering.
draw_echogram_object_overlay = True
echogram_object_overlay_linewidth = 0.7
echogram_object_label_every_n_frames = 10
echogram_object_tick_color = "white"
echogram_object_tick_linewidth = 0.8
echogram_object_tick_half_height = 2
echogram_frame_width_inches = 0.22
echogram_min_width_inches = 14
echogram_max_width_inches = 80
echogram_height_inches = 8
echogram_save_dpi = 200

os.makedirs(output_dir, exist_ok=True)
os.makedirs(diagnostics_dir, exist_ok=True)


In [ ]:
frame_fns_all = os.listdir(frame_dir)
frame_fns_all = [fn for fn in frame_fns_all if fn.endswith(".jpg") or fn.endswith(".png")]
frame_fns_all.sort()

print(f"Found {len(frame_fns_all)} frame(s) in {frame_dir}")
print("First frames:", frame_fns_all[:5])
print("Last frames:", frame_fns_all[-5:])

frame_window_fns = frame_fns_all[first_frame_number:last_frame_number]
if not frame_window_fns:
    raise ValueError(
        f"No frames found in selection window {first_frame_number}:{last_frame_number}"
    )

selected_frame_count = min(num_frames, len(frame_window_fns))
if frame_selection_mode == "first":
    frame_fns = frame_window_fns[:selected_frame_count]
elif frame_selection_mode == "evenly_sampled":
    frame_indices = np.linspace(
        0,
        len(frame_window_fns) - 1,
        selected_frame_count,
        dtype=int,
    )
    frame_fns = [frame_window_fns[i] for i in frame_indices]
else:
    raise ValueError(f"Unknown frame_selection_mode: {frame_selection_mode}")

if not frame_fns:
    raise ValueError("No frames found to process")

background_sample_size = min(background_sample_count, len(frame_window_fns))
background_sample_indices = np.linspace(
    0,
    len(frame_window_fns) - 1,
    background_sample_size,
    dtype=int,
)
background_frame_fns = [frame_window_fns[i] for i in background_sample_indices]
preview_frame_fns = frame_fns[:min(frames_to_plot, len(frame_fns))]

print(
    f"Selection window {first_frame_number}:{last_frame_number} contains "
    f"{len(frame_window_fns)} frame(s)"
)
print(f"Using {len(frame_fns)} frame(s) via {frame_selection_mode!r} mode")
print("Selected first frames:", frame_fns[:5])
print("Selected last frames:", frame_fns[-5:])
print(f"Using {len(background_frame_fns)} evenly spaced frame(s) from the selection window for background")
print(f"Keeping {len(preview_frame_fns)} preview frame(s) in memory")

In [ ]:
def read_warp_metadata(path):
    metadata = {}
    with open(path) as f:
        for line in f:
            if ":" not in line:
                continue
            key, value = line.strip().split(":", 1)
            metadata[key.strip()] = value.strip()
    return metadata


def metadata_float(metadata, key, default=None):
    value = metadata.get(key)
    if value is None or value == "unavailable":
        return default
    return float(value)


def frame_stem(frame_fn):
    stem = os.path.splitext(os.path.basename(frame_fn))[0]
    if stem.endswith("_raw_rotated"):
        stem = stem[: -len("_raw_rotated")]
    return stem


def metadata_path_for_frame(frame_fn):
    return os.path.join(os.path.dirname(frame_dir), f"{frame_stem(frame_fn)}_warp_metadata.txt")


def theta_path_for_frame(frame_fn):
    return os.path.join(os.path.dirname(frame_dir), f"{frame_stem(frame_fn)}_theta.csv")


def load_theta_degrees(path):
    theta_table = np.genfromtxt(path, delimiter=",", names=True)
    return np.asarray(theta_table["theta_degrees"], dtype=np.float32)


def load_polar_frame(frame_path):
    frame = cv2.imread(frame_path, cv2.IMREAD_GRAYSCALE)
    if frame is None:
        raise FileNotFoundError(f"Could not read frame: {frame_path}")
    return frame


def resolve_interpolation_mode(mode):
    interpolation_modes = {
        "nearest": cv2.INTER_NEAREST,
        "linear": cv2.INTER_LINEAR,
        "cubic": cv2.INTER_CUBIC,
        "area": cv2.INTER_AREA,
        "lanczos4": cv2.INTER_LANCZOS4,
    }
    normalized_mode = str(mode).lower()
    if normalized_mode not in interpolation_modes:
        options = ", ".join(sorted(interpolation_modes))
        raise ValueError(f"Unknown warp_interpolation_mode {mode!r}; choose one of: {options}")
    return interpolation_modes[normalized_mode]


def center_column_valid_span(polar_image, signal_threshold=center_column_signal_threshold):
    center_column = polar_image[:, polar_image.shape[1] // 2]
    signal_rows = np.flatnonzero(center_column > signal_threshold)
    if signal_rows.size == 0:
        raise ValueError("Center column has no signal above threshold; cannot infer selected range scale")
    first_valid_row = int(signal_rows[0])
    last_valid_row = int(signal_rows[-1])
    return first_valid_row, last_valid_row, last_valid_row - first_valid_row + 1


def build_warp_maps(
    polar_image,
    theta_degrees,
    selected_range_m,
    meters_per_range_bin,
    first_valid_range_row,
    last_valid_range_row,
    output_scale=1,
):
    range_count, theta_count = polar_image.shape
    if theta_degrees.size != theta_count:
        raise ValueError(
            f"Theta count ({theta_degrees.size}) does not match polar image columns ({theta_count})"
        )

    output_scale = float(output_scale)
    if output_scale <= 0:
        raise ValueError("warp_output_scale must be greater than 0")

    theta_min_rad = np.deg2rad(float(np.min(theta_degrees)))
    theta_max_rad = np.deg2rad(float(np.max(theta_degrees)))
    x_min_m = selected_range_m * np.sin(theta_min_rad)
    x_max_m = selected_range_m * np.sin(theta_max_rad)

    output_height = int(np.ceil((last_valid_range_row - first_valid_range_row + 1) * output_scale))
    output_width = int(np.ceil(((x_max_m - x_min_m) / meters_per_range_bin) * output_scale)) + 1
    x_coords = np.linspace(x_min_m, x_max_m, output_width, dtype=np.float32)
    # y_coords = np.linspace(0, selected_range_m, output_height, dtype=np.float32)
    y_coords = np.linspace(selected_range_m, 0, output_height, dtype=np.float32)
    x_grid, y_grid = np.meshgrid(x_coords, y_coords)

    range_m_map = np.sqrt(x_grid**2 + y_grid**2)
    range_bin_map = range_m_map / meters_per_range_bin
    source_range_map = last_valid_range_row - range_bin_map
    theta_map = np.rad2deg(np.arctan2(x_grid, y_grid))
    theta_index_map = np.interp(theta_map, theta_degrees, np.arange(theta_count)).astype(np.float32)

    outside_wedge = (
        (range_bin_map < 0)
        | (range_m_map > selected_range_m)
        | (source_range_map < first_valid_range_row)
        | (source_range_map > last_valid_range_row)
        | (theta_map < theta_degrees.min())
        | (theta_map > theta_degrees.max())
    )
    source_range_map = source_range_map.astype(np.float32)
    theta_index_map[outside_wedge] = -1
    source_range_map[outside_wedge] = -1
    return theta_index_map, source_range_map


warp_reference_frame_fn = frame_fns[0]
warp_reference_image = load_polar_frame(os.path.join(frame_dir, warp_reference_frame_fn))
warp_metadata = read_warp_metadata(metadata_path_for_frame(warp_reference_frame_fn))
warp_theta_degrees = load_theta_degrees(theta_path_for_frame(warp_reference_frame_fn))
selected_range_m = metadata_float(warp_metadata, "estimated_selected_range_meters")
selected_range_ft = metadata_float(warp_metadata, "estimated_selected_range_feet")
if selected_range_m is None:
    raise ValueError("Warp metadata must include estimated_selected_range_meters")

first_valid_range_row, last_valid_range_row, valid_range_rows = center_column_valid_span(warp_reference_image)
meters_per_range_bin = selected_range_m / max(valid_range_rows - 1, 1)
warp_x_map, warp_y_map = build_warp_maps(
    warp_reference_image,
    warp_theta_degrees,
    selected_range_m,
    meters_per_range_bin,
    first_valid_range_row,
    last_valid_range_row,
    output_scale=warp_output_scale,
)
warp_interpolation = resolve_interpolation_mode(warp_interpolation_mode)

print(
    f"Warping frames to cartesian {warp_x_map.shape} from center-column rows "
    f"{first_valid_range_row}..{last_valid_range_row} "
    f"({valid_range_rows} rows, threshold > {center_column_signal_threshold})"
)
print(f"Warp interpolation mode: {warp_interpolation_mode}")
print(f"Warp output scale: {warp_output_scale}x")
print(
    f"Warp source scale: {meters_per_range_bin * 3.280839895:.4f} ft/bin; "
    f"output scale: {(meters_per_range_bin * 3.280839895) / warp_output_scale:.4f} ft/pixel; "
    f"selected range: {selected_range_ft:.2f} ft"
)


def load_frame_channel(frame_path, channel=0):
    polar_frame = load_polar_frame(frame_path)
    if polar_frame.shape != warp_reference_image.shape:
        raise ValueError(f"Frame shape changed at {frame_path}: {polar_frame.shape} != {warp_reference_image.shape}")
    return cv2.remap(
        polar_frame,
        warp_x_map,
        warp_y_map,
        interpolation=warp_interpolation,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )


def processed_frame_blur_kernel():
    kernel_size = int(processed_frame_blur_kernel_size)
    if kernel_size <= 0:
        return None
    if kernel_size % 2 == 0:
        kernel_size += 1
    return (kernel_size, kernel_size)


def load_processed_frame(frame_fn):
    frame_path = os.path.join(frame_dir, frame_fn)
    frame = load_frame_channel(frame_path).astype(np.float32)
    blur_kernel = processed_frame_blur_kernel()
    if blur_kernel is None:
        return frame
    return cv2.GaussianBlur(frame, blur_kernel, 0)


frame_sum = None
frame_shape = None

for frame_fn in tqdm(background_frame_fns, desc="Building warped background"):
    frame = load_processed_frame(frame_fn)

    if frame_sum is None:
        frame_shape = frame.shape
        frame_sum = np.zeros(frame_shape, dtype=np.float64)
    elif frame.shape != frame_shape:
        raise ValueError(f"Frame shape changed at {frame_fn}: {frame.shape} != {frame_shape}")

    frame_sum += frame

if frame_sum is None:
    raise ValueError("No background frames found to process")


In [ ]:
frames_averaged = (frame_sum / len(background_frame_fns)).astype(np.float32)
del frame_sum


def odd_kernel_size(value, minimum=3):
    value = max(int(value), minimum)
    if value % 2 == 0:
        value += 1
    return value


def threshold_background_subtraction(bgs_float, bgs_frame):
    if threshold_mode == "fixed":
        return (bgs_float >= bgs_tol).astype(np.uint8) * 255

    if threshold_mode == "percentile":
        positive_pixels = bgs_float[bgs_float > 0]
        if positive_pixels.size == 0:
            return np.zeros_like(bgs_frame, dtype=np.uint8)
        threshold = max(bgs_tol, float(np.percentile(positive_pixels, bgs_percentile)))
        return (bgs_float >= threshold).astype(np.uint8) * 255

    if threshold_mode == "adaptive":
        block_size = odd_kernel_size(adaptive_threshold_block_size)
        adaptive_mask = cv2.adaptiveThreshold(
            bgs_frame,
            255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY,
            block_size,
            adaptive_threshold_c,
        )
        adaptive_mask[bgs_float < adaptive_min_bgs_tol] = 0
        return adaptive_mask

    raise ValueError(f"Unknown threshold_mode: {threshold_mode}")


def make_background_subtracted_frames(frame_fn):
    frame = load_processed_frame(frame_fn)
    if frame.shape != frame_shape:
        raise ValueError(f"Frame shape changed at {frame_fn}: {frame.shape} != {frame_shape}")

    bgs_float = np.clip(frame - frames_averaged, 0, 255).astype(np.float32) / 255
    bgs_frame = np.clip(bgs_float * 255, 0, 255).astype(np.uint8)
    bgs_tolled = threshold_background_subtraction(bgs_float, bgs_frame)
    return frame.astype(np.uint8), bgs_frame, bgs_tolled


preview_frames = []
preview_bgs_frames = []
preview_bgs_tolled = []
for frame_fn in tqdm(preview_frame_fns, desc="Preparing preview frames"):
    frame, bgs_frame, bgs_tolled = make_background_subtracted_frames(frame_fn)
    preview_frames.append(frame)
    preview_bgs_frames.append(bgs_frame)
    preview_bgs_tolled.append(bgs_tolled)

print(len(frame_fns), len(background_frame_fns), frame_shape)

display_count = min(frames_to_plot, len(preview_frames))
fig, ax = plt.subplots(2, display_count, figsize=(20, 10))
if display_count == 1:
    ax = np.array(ax).reshape(2, 1)

for i, (frame, bgs_frame) in enumerate(zip(preview_frames[:display_count], preview_bgs_frames[:display_count])):
    ax[0, i].imshow(frame, cmap="gray")
    ax[0, i].set_title(f"Frame {i}")
    ax[0, i].axis("off")
    print(type(bgs_frame), bgs_frame.shape, np.min(bgs_frame), np.max(bgs_frame))
    ax[1, i].imshow(bgs_frame, cmap="gray")
    ax[1, i].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# make preview GIFs of the frames and the background-subtracted frames
orig_frames = np.array(preview_frames, dtype=np.uint8)
bgs_frames = np.array(preview_bgs_frames, dtype=np.uint8)
bgs_frames_tolled = np.array(preview_bgs_tolled, dtype=np.uint8)

orig_rgb = [np.stack([f, f, f], axis=-1) for f in orig_frames]

# overlay the thresholded background subtraction as a tint on the original frames
orig_bgs_tolled_tinted_rgb = []
for orig, mask in zip(orig_rgb, bgs_frames_tolled):
    tinted = orig.astype(np.float32).copy()
    mask = mask > 0
    tinted[mask] = (1 - motion_tint_alpha) * tinted[mask] + motion_tint_alpha * motion_tint_color
    orig_bgs_tolled_tinted_rgb.append(np.clip(tinted, 0, 255).astype(np.uint8))

# make bgs in viridis color map for better visibility
bgs_rgb = [(plt.cm.viridis(f / 255.0)[:, :, :3] * 255).astype(np.uint8) for f in bgs_frames]
bgs_tolled_rgb = [(plt.cm.viridis(f / 255.0)[:, :, :3] * 255).astype(np.uint8) for f in bgs_frames_tolled]

orig_path = os.path.join(diagnostics_dir, preview_frames_gif_name)
bgs_path = os.path.join(diagnostics_dir, preview_bgs_gif_name)
bgs_tolled_path = os.path.join(diagnostics_dir, preview_bgs_tolled_gif_name)
orig_bgs_tolled_tinted_path = os.path.join(diagnostics_dir, preview_tinted_gif_name)

imageio.mimsave(orig_path, orig_rgb, duration=gif_duration, loop=0)
imageio.mimsave(bgs_path, bgs_rgb, duration=gif_duration, loop=0)
imageio.mimsave(bgs_tolled_path, bgs_tolled_rgb, duration=gif_duration, loop=0)
imageio.mimsave(orig_bgs_tolled_tinted_path, orig_bgs_tolled_tinted_rgb, duration=gif_duration, loop=0)

print("Saved diagnostics:", orig_path, bgs_path, bgs_tolled_path, orig_bgs_tolled_tinted_path)

display(IPImage(orig_path))
display(IPImage(bgs_path))
display(IPImage(bgs_tolled_path))
display(IPImage(orig_bgs_tolled_tinted_path))


In [ ]:
# SORT tracker state. Run this cell before rerunning the tracking loop.
object_tracks = {}
all_frame_detections = []
tracked_rgb = []
track_colors = {}
sort_tracker = None


In [ ]:
# SORT tracker helper functions.
def track_color(track_id):
    if track_id not in track_colors:
        track_colors[track_id] = track_color_palette[(track_id - 1) % len(track_color_palette)]
    return track_colors[track_id]


def save_gif_with_stable_id_palette(path, rgb_frames, duration=gif_duration):
    from PIL import Image

    grayscale_palette = [
        (int(value), int(value), int(value))
        for value in np.linspace(0, 255, 256 - len(track_color_palette))
    ]
    palette_colors = track_color_palette + grayscale_palette
    palette_values = [channel for color in palette_colors for channel in color]
    palette_values.extend([0] * (768 - len(palette_values)))

    palette_image = Image.new("P", (1, 1))
    palette_image.putpalette(palette_values)

    pil_frames = [
        Image.fromarray(frame, "RGB").quantize(
            palette=palette_image,
            dither=Image.Dither.NONE,
        )
        for frame in rgb_frames
    ]
    pil_frames[0].save(
        path,
        save_all=True,
        append_images=pil_frames[1:],
        duration=int(duration * 1000),
        loop=0,
        optimize=False,
        disposal=2,
    )


def union_bbox(bbox_a, bbox_b):
    ax, ay, aw, ah = bbox_a
    bx, by, bw, bh = bbox_b
    x1 = min(ax, bx)
    y1 = min(ay, by)
    x2 = max(ax + aw, bx + bw)
    y2 = max(ay + ah, by + bh)
    return (x1, y1, x2 - x1, y2 - y1)


def find_root(parents, node):
    while parents[node] != node:
        parents[node] = parents[parents[node]]
        node = parents[node]
    return node


def union_roots(parents, node_a, node_b):
    root_a = find_root(parents, node_a)
    root_b = find_root(parents, node_b)
    if root_a != root_b:
        parents[root_b] = root_a


def cluster_summary(group):
    area = sum(detection["area"] for detection in group)
    centroid = sum(
        detection["centroid"] * detection["area"] for detection in group
    ) / max(area, 1)
    confidence = sum(
        detection.get("confidence", 0.0) * detection["area"] for detection in group
    ) / max(area, 1)
    bbox = group[0]["bbox"]
    for detection in group[1:]:
        bbox = union_bbox(bbox, detection["bbox"])
    return int(area), centroid, bbox, float(confidence)


def equivalent_radius(area):
    return np.sqrt(max(float(area), 1.0) / np.pi)


def cluster_components(detections):
    if not detections:
        return []

    parents = list(range(len(detections)))
    for i, detection_a in enumerate(detections):
        for j in range(i + 1, len(detections)):
            detection_b = detections[j]
            distance = np.linalg.norm(detection_a["centroid"] - detection_b["centroid"])
            join_distance = max(
                cluster_distance,
                equivalent_radius(detection_a["area"]) + equivalent_radius(detection_b["area"]),
            )
            if distance <= join_distance:
                union_roots(parents, i, j)

    cluster_groups = {}
    for detection_index, detection in enumerate(detections):
        root = find_root(parents, detection_index)
        cluster_groups.setdefault(root, []).append(detection)

    clusters = []
    for group in cluster_groups.values():
        area, centroid, bbox, confidence = cluster_summary(group)
        clusters.append(
            {
                "centroid": centroid,
                "bbox": bbox,
                "area": area,
                "confidence": confidence,
                "parts": group,
            }
        )
    return clusters


def bbox_to_sort_z(bbox):
    x, y, w, h = bbox
    w = max(float(w), 1.0)
    h = max(float(h), 1.0)
    return np.array([x + w / 2.0, y + h / 2.0, w * h, w / h], dtype=np.float32)


def sort_x_to_bbox(state):
    x, y, s, r = state[:4]
    s = max(float(s), 1.0)
    r = max(float(r), 1e-3)
    w = np.sqrt(s * r)
    h = s / max(w, 1e-3)
    return np.array([x - w / 2.0, y - h / 2.0, x + w / 2.0, y + h / 2.0], dtype=np.float32)


def xywh_to_xyxy(bbox):
    x, y, w, h = bbox
    return np.array([x, y, x + w, y + h], dtype=np.float32)


def xyxy_to_xywh(bbox):
    x1, y1, x2, y2 = bbox
    return (
        int(round(x1)),
        int(round(y1)),
        max(1, int(round(x2 - x1))),
        max(1, int(round(y2 - y1))),
    )


def bbox_iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)
    iw = max(0.0, ix2 - ix1)
    ih = max(0.0, iy2 - iy1)
    intersection = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - intersection
    if union <= 0:
        return 0.0
    return float(intersection / union)


def normalized_confidence(value):
    return float(np.clip(value, 0.0, 255.0) / 255.0)


def normalized_area(value, reference_area):
    return float(value) / max(float(reference_area), 1.0)


def tracker_predicted_centroid(tracker):
    return np.array([tracker.x[0, 0], tracker.x[1, 0]], dtype=np.float32)


def trajectory_score(detection, tracker):
    distance = np.linalg.norm(detection["centroid"] - tracker_predicted_centroid(tracker))
    return max(0.0, 1.0 - float(distance) / max(float(sort_trajectory_max_distance), 1.0))


def trajectory_score_matrix(detections, trackers):
    score_matrix = np.zeros((len(detections), len(trackers)), dtype=np.float32)
    for detection_index, detection in enumerate(detections):
        for tracker_index, tracker in enumerate(trackers):
            score_matrix[detection_index, tracker_index] = trajectory_score(detection, tracker)
    return score_matrix


def tracker_candidates_for_detection(detection_index, iou_matrix, trajectory_matrix, method):
    if method == "trajectory":
        return np.flatnonzero(trajectory_matrix[detection_index] >= sort_trajectory_min_score)
    return np.flatnonzero(iou_matrix[detection_index] >= sort_collision_iou_threshold)


def collision_candidate_trackers(iou_matrix, trajectory_matrix, method):
    collision_trackers = set()
    for detection_index in range(iou_matrix.shape[0]):
        candidate_trackers = tracker_candidates_for_detection(
            detection_index, iou_matrix, trajectory_matrix, method
        )
        if candidate_trackers.size >= 2:
            collision_trackers.update(int(tracker_index) for tracker_index in candidate_trackers)
    return collision_trackers


def collision_score_matrix(detections, iou_matrix, trajectory_matrix, trackers):
    method = str(sort_collision_tiebreaker_method).strip().lower()
    valid_methods = {"iou", "confidence", "area", "trajectory"}
    if method not in valid_methods:
        options = ", ".join(sorted(valid_methods))
        raise ValueError(
            f"Unknown sort_collision_tiebreaker_method {sort_collision_tiebreaker_method!r}; "
            f"choose one of: {options}"
        )

    if method == "trajectory":
        return trajectory_matrix + sort_trajectory_iou_tiebreaker * iou_matrix

    score_matrix = iou_matrix.copy()
    if method == "iou":
        return score_matrix

    for detection_index in range(iou_matrix.shape[0]):
        candidate_trackers = tracker_candidates_for_detection(
            detection_index, iou_matrix, trajectory_matrix, method
        )
        if candidate_trackers.size < 2:
            continue

        if method == "area":
            reference_area = max(
                trackers[tracker_index].area for tracker_index in candidate_trackers
            )

        for tracker_index in candidate_trackers:
            if method == "confidence":
                primary_score = normalized_confidence(trackers[tracker_index].confidence)
            else:
                primary_score = normalized_area(trackers[tracker_index].area, reference_area)

            score_matrix[detection_index, tracker_index] = (
                primary_score
                + sort_collision_iou_tiebreaker * float(iou_matrix[detection_index, tracker_index])
            )
    return score_matrix


def exact_max_assignment(score_matrix):
    rows, cols = score_matrix.shape
    if rows == 0 or cols == 0:
        return []

    transposed = False
    matrix = score_matrix
    if cols < rows:
        matrix = score_matrix.T
        rows, cols = matrix.shape
        transposed = True

    dp = {0: (0.0, [])}
    for row in range(rows):
        next_dp = {}
        for used_mask, (score, pairs) in dp.items():
            current = next_dp.get(used_mask)
            if current is None or score > current[0]:
                next_dp[used_mask] = (score, pairs)
            for col in range(cols):
                bit = 1 << col
                if used_mask & bit:
                    continue
                new_mask = used_mask | bit
                new_score = score + float(matrix[row, col])
                new_pairs = pairs + [(row, col)]
                current = next_dp.get(new_mask)
                if current is None or new_score > current[0]:
                    next_dp[new_mask] = (new_score, new_pairs)
        dp = next_dp

    best_pairs = max(dp.values(), key=lambda item: item[0])[1]
    if transposed:
        return [(col, row) for row, col in best_pairs]
    return best_pairs


def greedy_max_assignment(score_matrix):
    pairs = []
    used_rows = set()
    used_cols = set()
    candidates = [
        (float(score_matrix[row, col]), row, col)
        for row in range(score_matrix.shape[0])
        for col in range(score_matrix.shape[1])
    ]
    for _, row, col in sorted(candidates, reverse=True):
        if row in used_rows or col in used_cols:
            continue
        used_rows.add(row)
        used_cols.add(col)
        pairs.append((row, col))
    return pairs


def assign_detections_to_trackers(detections, trackers, iou_threshold=sort_iou_threshold):
    if len(trackers) == 0:
        return [], list(range(len(detections))), [], set()

    iou_matrix = np.zeros((len(detections), len(trackers)), dtype=np.float32)
    for detection_index, detection in enumerate(detections):
        detection_bbox = xywh_to_xyxy(detection["bbox"])
        for tracker_index, tracker in enumerate(trackers):
            iou_matrix[detection_index, tracker_index] = bbox_iou(detection_bbox, tracker.predicted_bbox)

    method = str(sort_collision_tiebreaker_method).strip().lower()
    trajectory_matrix = trajectory_score_matrix(detections, trackers)
    score_matrix = collision_score_matrix(detections, iou_matrix, trajectory_matrix, trackers)
    collision_trackers = collision_candidate_trackers(iou_matrix, trajectory_matrix, method)

    exact_size = max(score_matrix.shape)
    if exact_size <= sort_exact_assignment_limit:
        candidate_matches = exact_max_assignment(score_matrix)
    else:
        candidate_matches = greedy_max_assignment(score_matrix)

    matches = []
    unmatched_detections = set(range(len(detections)))
    unmatched_trackers = set(range(len(trackers)))
    for detection_index, tracker_index in candidate_matches:
        enough_iou = iou_matrix[detection_index, tracker_index] >= iou_threshold
        enough_trajectory = (
            method == "trajectory"
            and trajectory_matrix[detection_index, tracker_index] >= sort_trajectory_min_score
        )
        if not (enough_iou or enough_trajectory):
            continue
        matches.append((detection_index, tracker_index))
        unmatched_detections.discard(detection_index)
        unmatched_trackers.discard(tracker_index)

    return matches, sorted(unmatched_detections), sorted(unmatched_trackers), collision_trackers


class KalmanBoxTracker:
    count = 0

    def __init__(self, detection):
        KalmanBoxTracker.count += 1
        self.id = KalmanBoxTracker.count
        self.x = np.zeros((7, 1), dtype=np.float32)
        self.x[:4, 0] = bbox_to_sort_z(detection["bbox"])
        self.P = np.eye(7, dtype=np.float32)
        self.P[4:, 4:] *= 1000.0
        self.P *= 10.0
        self.F = np.eye(7, dtype=np.float32)
        self.F[0, 4] = 1.0
        self.F[1, 5] = 1.0
        self.F[2, 6] = 1.0
        self.H = np.zeros((4, 7), dtype=np.float32)
        self.H[0, 0] = 1.0
        self.H[1, 1] = 1.0
        self.H[2, 2] = 1.0
        self.H[3, 3] = 1.0
        self.R = np.eye(4, dtype=np.float32)
        self.R[2:, 2:] *= 10.0
        self.Q = np.eye(7, dtype=np.float32) * 0.01
        self.Q[4:, 4:] *= 0.01
        self.time_since_update = 0
        self.hits = 1
        self.hit_streak = 1
        self.age = 0
        self.last_detection = detection
        self.confidence = float(detection.get("confidence", 0.0))
        self.area = float(detection.get("area", 0.0))
        self.predicted_bbox = xywh_to_xyxy(detection["bbox"])

    def predict(self):
        if self.x[2, 0] + self.x[6, 0] <= 1.0:
            self.x[6, 0] = 0.0
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q
        self.age += 1
        if self.time_since_update > 0:
            self.hit_streak = 0
        self.time_since_update += 1
        self.predicted_bbox = sort_x_to_bbox(self.x[:, 0])
        return self.predicted_bbox

    def update(self, detection):
        z = bbox_to_sort_z(detection["bbox"]).reshape((4, 1))
        y = z - self.H @ self.x
        s = self.H @ self.P @ self.H.T + self.R
        k = self.P @ self.H.T @ np.linalg.inv(s)
        self.x = self.x + k @ y
        i = np.eye(7, dtype=np.float32)
        self.P = (i - k @ self.H) @ self.P
        self.time_since_update = 0
        self.hits += 1
        self.hit_streak += 1
        self.last_detection = detection
        self.confidence = float(detection.get("confidence", self.confidence))
        self.area = float(detection.get("area", self.area))
        self.predicted_bbox = sort_x_to_bbox(self.x[:, 0])

    def mark_collision_occluded(self):
        self.time_since_update = 0
        self.hit_streak = 0


class SortTracker:
    def __init__(self, max_age=sort_max_age, min_hits=sort_min_hits, iou_threshold=sort_iou_threshold):
        KalmanBoxTracker.count = 0
        self.max_age = max_age
        self.min_hits = min_hits
        self.iou_threshold = iou_threshold
        self.trackers = []
        self.frame_count = 0

    def update(self, detections):
        self.frame_count += 1
        for tracker in self.trackers:
            tracker.predict()

        matches, unmatched_detections, unmatched_trackers, collision_trackers = assign_detections_to_trackers(
            detections,
            self.trackers,
            self.iou_threshold,
        )

        matched_tracker_indices = {tracker_index for _, tracker_index in matches}
        if sort_keep_collision_tracks_alive:
            for tracker_index in collision_trackers - matched_tracker_indices:
                if tracker_index in unmatched_trackers:
                    self.trackers[tracker_index].mark_collision_occluded()

        matched_detections = []
        for detection_index, tracker_index in matches:
            tracker = self.trackers[tracker_index]
            detection = detections[detection_index]
            tracker.update(detection)
            matched_detections.append({"track_id": tracker.id, **detection})

        for detection_index in unmatched_detections:
            tracker = KalmanBoxTracker(detections[detection_index])
            self.trackers.append(tracker)
            matched_detections.append({"track_id": tracker.id, **detections[detection_index]})

        self.trackers = [
            tracker for tracker in self.trackers
            if tracker.time_since_update <= self.max_age
        ]

        matched_detections.sort(key=lambda detection: detection["track_id"])
        return matched_detections


In [ ]:
# Run blob detection and SORT motion tracking, streaming frames one at a time.
sort_tracker = SortTracker(
    max_age=sort_max_age,
    min_hits=sort_min_hits,
    iou_threshold=sort_iou_threshold,
)
object_tracks = {}
all_frame_detections = []

for frame_index, frame_fn in enumerate(tqdm(frame_fns, desc="SORT tracking frames")):
    _, bgs_frame, mask = make_background_subtracted_frames(frame_fn)

    binary_mask = (mask > 0).astype(np.uint8)
    kernel = np.ones((morph_kernel_size, morph_kernel_size), np.uint8)
    binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_CLOSE, kernel)
    binary_mask = cv2.dilate(binary_mask, kernel, iterations=1)

    component_count, labels, stats, centroids = cv2.connectedComponentsWithStats(
        binary_mask, connectivity=8
    )

    detections = []
    for component_id in range(1, component_count):
        x, y, w, h, area = stats[component_id]
        if area < min_blob_area:
            continue
        cx, cy = centroids[component_id]
        component_pixels = labels == component_id
        confidence = float(np.mean(bgs_frame[component_pixels])) if np.any(component_pixels) else 0.0
        detections.append(
            {
                "component_id": int(component_id),
                "centroid": np.array([cx, cy], dtype=np.float32),
                "bbox": (int(x), int(y), int(w), int(h)),
                "area": int(area),
                "confidence": confidence,
            }
        )

    clusters = cluster_components(detections)
    frame_detections = sort_tracker.update(sorted(clusters, key=lambda item: item["area"], reverse=True))

    for detection in frame_detections:
        track_id = detection["track_id"]
        object_tracks.setdefault(track_id, [])
        object_tracks[track_id].append(
            {
                "frame": frame_index,
                "centroid": tuple(detection["centroid"]),
                "bbox": detection["bbox"],
                "area": detection["area"],
                "confidence": detection.get("confidence", 0.0),
                "part_count": len(detection["parts"]),
            }
        )

    all_frame_detections.append(frame_detections)


In [ ]:
# Filter tracks and remap raw IDs to compact display IDs.
raw_object_tracks = object_tracks
raw_all_frame_detections = all_frame_detections
rejected_track_reasons = {}
track_tiny_flicker_stats_by_id = {}


def track_tiny_flicker_rejection_reasons(track):
    reasons = []
    frames_present = {observation["frame"] for observation in track}
    areas = np.array([observation["area"] for observation in track], dtype=np.float32)

    if areas.size == 0:
        stats = {
            "frames": int(len(frames_present)),
            "observations": 0,
            "median_area": 0.0,
            "max_area": 0.0,
            "large_detection_fraction": 0.0,
            "short_small_flash": False,
        }
        return ["no_observations"], stats

    median_area = float(np.median(areas))
    max_area = float(np.max(areas))
    large_detection_fraction = float(np.mean(areas >= large_detection_area))
    stats = {
        "frames": int(len(frames_present)),
        "observations": int(len(track)),
        "median_area": median_area,
        "max_area": max_area,
        "large_detection_fraction": large_detection_fraction,
    }

    short_small_flash = (
        len(frames_present) <= max_short_flash_frames
        and median_area <= max_short_flash_median_area
        and max_area <= max_short_flash_area
    )
    stats["short_small_flash"] = bool(short_small_flash)
    if reject_short_flash_tracks and short_small_flash:
        reasons.append("short_small_flash")

    has_large_detection = max_area >= large_detection_area
    if len(frames_present) < min_track_frames:
        reasons.append("too_few_frames")

    if reject_tiny_flicker_tracks and not has_large_detection:
        if median_area < min_track_median_area:
            reasons.append("tiny_median_area")
        if large_detection_fraction < min_large_detection_fraction:
            reasons.append("mostly_tiny_detections")

    return reasons, stats


object_tracks = {}
for track_id, track in raw_object_tracks.items():
    reasons, stats = track_tiny_flicker_rejection_reasons(track)
    track_tiny_flicker_stats_by_id[track_id] = stats
    if reasons:
        rejected_track_reasons[track_id] = reasons
        continue
    object_tracks[track_id] = track

valid_track_ids = set(object_tracks)
raw_to_display_track_id = {
    raw_track_id: display_track_id
    for display_track_id, raw_track_id in enumerate(sorted(valid_track_ids), start=1)
}
display_to_raw_track_id = {
    display_track_id: raw_track_id
    for raw_track_id, display_track_id in raw_to_display_track_id.items()
}
object_tracks = {
    raw_to_display_track_id[raw_track_id]: [
        {**observation, "raw_track_id": raw_track_id}
        for observation in track
    ]
    for raw_track_id, track in object_tracks.items()
}
all_frame_detections = [
    [
        {
            **detection,
            "raw_track_id": detection["track_id"],
            "track_id": raw_to_display_track_id[detection["track_id"]],
        }
        for detection in frame_detections
        if detection["track_id"] in valid_track_ids
    ]
    for frame_detections in raw_all_frame_detections
]

tracking_csv_path = os.path.join(output_dir, tracking_csv_name)
with open(tracking_csv_path, "w", newline="") as csv_file:
    fieldnames = [
        "frame_index",
        "frame_file",
        "track_id",
        "raw_track_id",
        "centroid_x",
        "centroid_y",
        "bbox_x",
        "bbox_y",
        "bbox_w",
        "bbox_h",
        "area",
        "confidence",
        "part_count",
        "component_ids",
    ]
    writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
    writer.writeheader()

    for frame_index, frame_detections in enumerate(all_frame_detections):
        frame_file = frame_fns[frame_index] if frame_index < len(frame_fns) else ""
        for detection in frame_detections:
            cx, cy = detection["centroid"]
            x, y, w, h = detection["bbox"]
            parts = detection.get("parts", [])
            writer.writerow(
                {
                    "frame_index": frame_index,
                    "frame_file": frame_file,
                    "track_id": detection["track_id"],
                    "raw_track_id": detection.get("raw_track_id", detection["track_id"]),
                    "centroid_x": float(cx),
                    "centroid_y": float(cy),
                    "bbox_x": int(x),
                    "bbox_y": int(y),
                    "bbox_w": int(w),
                    "bbox_h": int(h),
                    "area": int(detection["area"]),
                    "confidence": float(detection.get("confidence", 0.0)),
                    "part_count": len(parts),
                    "component_ids": ";".join(str(part.get("component_id", "")) for part in parts),
                }
            )

print(f"Saved tracking CSV: {tracking_csv_path}")


In [ ]:
# Plot per-track summary bar graphs.
def track_observations_sorted(track):
    return sorted(track, key=lambda observation: observation["frame"])


def track_summary_metrics(track):
    observations = track_observations_sorted(track)
    frames = np.array([observation["frame"] for observation in observations], dtype=np.float32)
    centroids = np.array([observation["centroid"] for observation in observations], dtype=np.float32)
    areas = np.array([observation["area"] for observation in observations], dtype=np.float32)

    if len(observations) < 2:
        median_speed = 0.0
        mean_speed = 0.0
    else:
        frame_deltas = np.diff(frames)
        centroid_deltas = np.linalg.norm(np.diff(centroids, axis=0), axis=1)
        valid = frame_deltas > 0
        speeds = centroid_deltas[valid] / frame_deltas[valid]
        if speeds.size:
            median_speed = float(np.median(speeds))
            mean_speed = float(np.mean(speeds))
        else:
            median_speed = 0.0
            mean_speed = 0.0

    return {
        "frames": int(len(set(int(frame) for frame in frames))),
        "observations": int(len(observations)),
        "median_area": float(np.median(areas)) if areas.size else 0.0,
        "mean_area": float(np.mean(areas)) if areas.size else 0.0,
        "max_area": float(np.max(areas)) if areas.size else 0.0,
        "median_speed_px_per_frame": median_speed,
        "mean_speed_px_per_frame": mean_speed,
    }


track_summary_by_id = {
    track_id: track_summary_metrics(track)
    for track_id, track in sorted(object_tracks.items())
}


def track_bar_color(track_id):
    return np.array(track_color(track_id), dtype=np.float32) / 255


def save_track_bar_graph(metric_key, title, ylabel, out_name):
    track_ids = list(track_summary_by_id)
    values = [track_summary_by_id[track_id][metric_key] for track_id in track_ids]
    colors = [track_bar_color(track_id) for track_id in track_ids]

    fig_width = max(12, min(60, len(track_ids) * 0.18))
    fig, ax = plt.subplots(figsize=(fig_width, 5))
    ax.bar(range(len(track_ids)), values, color=colors, width=0.9)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("Track ID")

    if track_ids:
        tick_step = max(1, int(np.ceil(len(track_ids) / 40)))
        tick_positions = list(range(0, len(track_ids), tick_step))
        ax.set_xticks(tick_positions)
        ax.set_xticklabels([str(track_ids[index]) for index in tick_positions], rotation=90)

    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()

    out_path = os.path.join(diagnostics_dir, out_name)
    fig.savefig(out_path, dpi=200)
    plt.show()
    plt.close(fig)
    return out_path


def save_track_length_area_scatter(out_name):
    track_ids = list(track_summary_by_id)
    lengths = [track_summary_by_id[track_id]["frames"] for track_id in track_ids]
    median_areas = [track_summary_by_id[track_id]["median_area"] for track_id in track_ids]
    colors = [track_bar_color(track_id) for track_id in track_ids]

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(lengths, median_areas, c=colors, s=28, alpha=0.85, edgecolors="black", linewidths=0.25)
    ax.set_title("Track length vs median segmentation area")
    ax.set_xlabel("Track length (frames observed)")
    ax.set_ylabel("Median segmentation area (pixels)")
    ax.grid(alpha=0.25)

    for track_id, x_value, y_value in zip(track_ids, lengths, median_areas):
        ax.annotate(str(track_id), (x_value, y_value), fontsize=5, alpha=0.65)

    fig.tight_layout()

    out_path = os.path.join(diagnostics_dir, out_name)
    fig.savefig(out_path, dpi=200)
    plt.show()
    plt.close(fig)
    return out_path


track_length_bar_path = save_track_bar_graph(
    "frames",
    "Track length by ID",
    "Frames observed",
    track_length_bar_name,
)
track_area_bar_path = save_track_bar_graph(
    "median_area",
    "Median segmentation area by ID",
    "Median area (pixels)",
    track_area_bar_name,
)
track_speed_bar_path = save_track_bar_graph(
    "median_speed_px_per_frame",
    "Median speed by ID",
    "Median speed (pixels / sampled frame)",
    track_speed_bar_name,
)
track_length_area_scatter_path = save_track_length_area_scatter(track_length_area_scatter_name)

print("Saved track summary plots:", track_length_bar_path, track_area_bar_path, track_speed_bar_path, track_length_area_scatter_path)
display(IPImage(track_length_bar_path))
display(IPImage(track_area_bar_path))
display(IPImage(track_speed_bar_path))
display(IPImage(track_length_area_scatter_path))


In [ ]:
# Render tracking GIF outputs, streaming frame images from disk.
def labels_for_tracking_mask(mask):
    binary_mask = (mask > 0).astype(np.uint8)
    kernel = np.ones((morph_kernel_size, morph_kernel_size), np.uint8)
    binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_CLOSE, kernel)
    binary_mask = cv2.dilate(binary_mask, kernel, iterations=1)
    _, labels, _, _ = cv2.connectedComponentsWithStats(binary_mask, connectivity=8)
    return binary_mask > 0, labels


def segmentation_overlay_for_frame(mask, frame_detections):
    segmented_pixels, labels = labels_for_tracking_mask(mask)
    overlay = np.zeros((*mask.shape, 3), dtype=np.uint8)
    assigned_pixels = np.zeros(mask.shape, dtype=bool)

    for detection in frame_detections:
        detection_pixels = np.zeros(mask.shape, dtype=bool)
        for part in detection.get("parts", []):
            component_id = part.get("component_id")
            if component_id is None:
                continue
            detection_pixels |= labels == component_id

        if not np.any(detection_pixels):
            continue

        assigned_pixels |= detection_pixels
        overlay[detection_pixels] = track_color(detection["track_id"])

    unassigned_pixels = segmented_pixels & ~assigned_pixels
    overlay[unassigned_pixels] = unassigned_segmentation_color
    overlay_pixels = assigned_pixels | unassigned_pixels
    return overlay, overlay_pixels


def apply_segmentation_overlay(base_rgb_frame, mask, frame_detections, alpha=segmentation_overlay_alpha):
    overlay, overlay_pixels = segmentation_overlay_for_frame(mask, frame_detections)
    annotated = base_rgb_frame.astype(np.float32).copy()
    annotated[overlay_pixels] = (1 - alpha) * annotated[overlay_pixels] + alpha * overlay[overlay_pixels]
    return np.clip(annotated, 0, 255).astype(np.uint8)


def draw_tracking_overlay_frame(base_rgb_frame, frame_detections):
    annotated = base_rgb_frame.copy()
    for detection in frame_detections:
        track_id = detection["track_id"]
        x, y, w, h = detection["bbox"]
        cx, cy = detection["centroid"]
        color = track_color(track_id)
        cv2.rectangle(annotated, (x, y), (x + w, y + h), color, 2)
        cv2.circle(annotated, (int(cx), int(cy)), 3, color, -1)
        cv2.putText(
            annotated,
            f"ID {track_id}",
            (x, max(12, y - 4)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.4,
            color,
            1,
            cv2.LINE_8,
        )
    return annotated


tracked_path = os.path.join(output_dir, tracked_gif_name)
tracked_raw_path = os.path.join(output_dir, tracked_raw_gif_name)
tracked_rgb = []
tracked_raw_rgb = []

with imageio.get_writer(tracked_path, mode="I", duration=gif_duration, loop=0) as tracked_writer,      imageio.get_writer(tracked_raw_path, mode="I", duration=gif_duration, loop=0) as raw_writer:
    for frame_index, (frame_fn, frame_detections) in enumerate(
        tqdm(zip(frame_fns, all_frame_detections), total=len(all_frame_detections), desc="Rendering tracking GIFs")
    ):
        orig_frame, bgs_frame, bgs_tolled = make_background_subtracted_frames(frame_fn)
        orig_rgb_frame = np.stack([orig_frame, orig_frame, orig_frame], axis=-1)
        bgs_rgb_frame = (plt.cm.viridis(bgs_frame / 255.0)[:, :, :3] * 255).astype(np.uint8)

        segmented_bgs = apply_segmentation_overlay(bgs_rgb_frame, bgs_tolled, frame_detections)
        segmented_raw = apply_segmentation_overlay(orig_rgb_frame, bgs_tolled, frame_detections)

        tracked_frame = draw_tracking_overlay_frame(segmented_bgs, frame_detections)
        tracked_raw_frame = draw_tracking_overlay_frame(segmented_raw, frame_detections)

        tracked_writer.append_data(tracked_frame)
        raw_writer.append_data(tracked_raw_frame)

        if frame_index < frames_to_plot:
            tracked_rgb.append(tracked_frame)
            tracked_raw_rgb.append(tracked_raw_frame)

print(
    f"Tracked {len(object_tracks)} object(s) after filtering "
    f"{len(raw_object_tracks)} raw track(s). "
    f"Rejected {len(rejected_track_reasons)} tiny/flicker track(s). "
    f"Saved: {tracked_path}, {tracked_raw_path}"
)
print(f"Display to raw track IDs: {display_to_raw_track_id}")
print(f"Rejected track reasons: {rejected_track_reasons}")
print(f"Unassigned segmentation color: {unassigned_segmentation_color}")
display(IPImage(tracked_path))
display(IPImage(tracked_raw_path))


In [ ]:
# Build thresholded and raw echograms from exact accepted tracker components.
# The thresholded echogram ignores motion that was not assigned to a kept ID.
tracked_echogram = np.zeros((frame_shape[0], len(all_frame_detections), 2), dtype=np.uint8)
raw_echogram = np.zeros((frame_shape[0], len(all_frame_detections), 2), dtype=np.uint8)
tracked_id_colour_echogram = np.zeros((frame_shape[0], len(all_frame_detections), 3), dtype=np.float32)
tracked_echogram_object_spans = []
tracker_kernel = np.ones((morph_kernel_size, morph_kernel_size), np.uint8)


def track_color_for_matplotlib(track_id):
    return np.array(track_color(track_id), dtype=np.float32) / 255


def echogram_angle_colour(echogram, brightness_power=1.5):
    angle_colour = echogram[:, :, 1] / 255
    angle_colour = plt.get_cmap("RdBu")(angle_colour)[:, :, :3]
    brightness = echogram[:, :, 0, None] / 255
    if np.max(brightness) > 0:
        brightness = brightness / np.max(brightness)
    brightness = brightness ** brightness_power
    return angle_colour * brightness


def echogram_figure_size(frame_count):
    width = frame_count * echogram_frame_width_inches
    width = min(max(width, echogram_min_width_inches), echogram_max_width_inches)
    return width, echogram_height_inches


def save_stretched_echogram_image(image, out_path, cmap=None):
    fig, ax = plt.subplots(figsize=echogram_figure_size(image.shape[1]))
    ax.imshow(image, aspect="auto", interpolation="nearest", cmap=cmap)
    ax.set_axis_off()
    fig.tight_layout(pad=0)
    fig.savefig(out_path, dpi=echogram_save_dpi, bbox_inches="tight", pad_inches=0)
    plt.show()
    plt.close(fig)


def draw_object_span_overlay(ax, base_shape):
    labeled_track_ids = set()
    for span in tracked_echogram_object_spans:
        object_color = track_color_for_matplotlib(span["track_id"])
        ax.vlines(
            span["frame"],
            span["y_min"],
            span["y_max"],
            colors=[object_color],
            linewidth=echogram_object_overlay_linewidth,
        )
        ax.vlines(
            span["frame"],
            max(0, span["y_centroid"] - echogram_object_tick_half_height),
            min(base_shape[0] - 1, span["y_centroid"] + echogram_object_tick_half_height),
            colors=echogram_object_tick_color,
            linewidth=echogram_object_tick_linewidth,
        )
        should_label = (
            span["track_id"] not in labeled_track_ids
            or span["frame"] % echogram_object_label_every_n_frames == 0
        )
        if should_label:
            ax.text(
                span["frame"],
                span["y_centroid"],
                str(span["track_id"]),
                color="white",
                fontsize=6,
                ha="center",
                va="center",
            )
            labeled_track_ids.add(span["track_id"])


def save_echogram_with_detection_overlay(base_image, out_path, detection_overlay=None):
    fig, ax = plt.subplots(figsize=echogram_figure_size(base_image.shape[1]))
    ax.imshow(base_image, aspect="auto", interpolation="nearest")
    if detection_overlay is not None:
        overlay_alpha = np.any(detection_overlay > 0, axis=2).astype(np.float32) * 0.85
        ax.imshow(detection_overlay, aspect="auto", interpolation="nearest", alpha=overlay_alpha)
    draw_object_span_overlay(ax, base_image.shape)
    ax.set_axis_off()
    fig.tight_layout(pad=0)
    fig.savefig(out_path, dpi=echogram_save_dpi, bbox_inches="tight", pad_inches=0)
    plt.show()
    plt.close(fig)


tracked_echogram_masks_path = os.path.join(diagnostics_dir, tracked_echogram_masks_name)
with imageio.get_writer(tracked_echogram_masks_path, mode="I", duration=gif_duration, loop=0) as mask_writer:
    for frame_index, (frame_fn, frame_detections) in enumerate(
        tqdm(zip(frame_fns, all_frame_detections), total=len(all_frame_detections), desc="Generating tracked echograms")
    ):
        orig_frame, _, bgs_tolled = make_background_subtracted_frames(frame_fn)
        tracked_mask = np.zeros_like(bgs_tolled, dtype=np.uint8)

        raw_frame_float = orig_frame.astype(np.float32) / 255
        raw_echogram_brightness = np.max(raw_frame_float, axis=1)
        raw_echogram_angle = np.argmax(raw_frame_float, axis=1) / raw_frame_float.shape[1]
        raw_echogram[:, frame_index, 0] = np.clip(raw_echogram_brightness * 255, 0, 255).astype(np.uint8)
        raw_echogram[:, frame_index, 1] = np.clip(raw_echogram_angle * 255, 0, 255).astype(np.uint8)

        binary_mask = (bgs_tolled > 0).astype(np.uint8)
        binary_mask = cv2.morphologyEx(binary_mask, cv2.MORPH_CLOSE, tracker_kernel)
        binary_mask = cv2.dilate(binary_mask, tracker_kernel, iterations=1)
        _, labels, _, _ = cv2.connectedComponentsWithStats(binary_mask, connectivity=8)

        for detection in frame_detections:
            component_pixels = np.zeros_like(tracked_mask, dtype=bool)
            for part in detection.get("parts", []):
                component_id = part.get("component_id")
                if component_id is None:
                    continue
                component_pixels |= labels == component_id

            tracked_mask[component_pixels] = 1
            ys = np.flatnonzero(np.any(component_pixels, axis=1))
            if ys.size:
                tracked_id_colour_echogram[ys, frame_index, :] = track_color_for_matplotlib(detection["track_id"])
                tracked_echogram_object_spans.append(
                    {
                        "frame": frame_index,
                        "track_id": detection["track_id"],
                        "y_min": int(ys[0]),
                        "y_max": int(ys[-1]),
                        "y_centroid": float(detection["centroid"][1]),
                    }
                )

        mask_writer.append_data((tracked_mask * 255).astype(np.uint8))

        echogram_brightness = np.max(tracked_mask, axis=1)
        echogram_angle = np.argmax(tracked_mask, axis=1) / tracked_mask.shape[1]

        tracked_echogram[:, frame_index, 0] = echogram_brightness * 255
        tracked_echogram[:, frame_index, 1] = echogram_angle * 255

plt.imshow(tracked_echogram[:, :, 0], cmap="gray")
plt.show()
plt.imshow(tracked_echogram[:, :, 1], cmap="RdBu")
plt.show()
plt.imshow(raw_echogram[:, :, 0], cmap="gray")
plt.show()
plt.imshow(raw_echogram[:, :, 1], cmap="RdBu")
plt.show()

tracked_angle_colour = echogram_angle_colour(tracked_echogram)
raw_angle_colour = echogram_angle_colour(raw_echogram, brightness_power=1.0)

plt.imshow(tracked_angle_colour)
plt.show()
plt.imshow(raw_angle_colour)
plt.show()
plt.imshow(tracked_id_colour_echogram)
plt.show()

tracked_echogram_out_path = os.path.join(output_dir, tracked_echogram_name)
tracked_echogram_objects_path = os.path.join(output_dir, tracked_echogram_objects_name)
tracked_raw_echogram_out_path = os.path.join(output_dir, tracked_raw_echogram_name)
tracked_raw_echogram_objects_path = os.path.join(output_dir, tracked_raw_echogram_objects_name)
tracked_echogram_brightness_path = os.path.join(diagnostics_dir, tracked_echogram_brightness_name)
tracked_raw_echogram_brightness_path = os.path.join(diagnostics_dir, tracked_raw_echogram_brightness_name)

save_stretched_echogram_image(tracked_angle_colour, tracked_echogram_out_path)
save_stretched_echogram_image(raw_angle_colour, tracked_raw_echogram_out_path)
save_stretched_echogram_image(tracked_echogram[:, :, 0], tracked_echogram_brightness_path, cmap="gray")
save_stretched_echogram_image(raw_echogram[:, :, 0], tracked_raw_echogram_brightness_path, cmap="gray")

if draw_echogram_object_overlay:
    save_echogram_with_detection_overlay(tracked_id_colour_echogram, tracked_echogram_objects_path)
    save_echogram_with_detection_overlay(
        raw_angle_colour,
        tracked_raw_echogram_objects_path,
        detection_overlay=tracked_id_colour_echogram,
    )

print(
    "Saved:",
    tracked_echogram_out_path,
    tracked_echogram_objects_path,
    tracked_raw_echogram_out_path,
    tracked_raw_echogram_objects_path,
    tracked_echogram_brightness_path,
    tracked_raw_echogram_brightness_path,
    tracked_echogram_masks_path,
)
display(IPImage(tracked_echogram_out_path))
display(IPImage(tracked_echogram_objects_path))
display(IPImage(tracked_raw_echogram_out_path))
display(IPImage(tracked_raw_echogram_objects_path))
display(IPImage(tracked_echogram_masks_path))
